In [2]:
# setup
import re, csv, html, time, requests
from pathlib import Path
from urllib.parse import urljoin, quote

out = Path("data/labins-index")
out.mkdir(parents=True, exist_ok=True)
logf = out / "log.csv"
page = "https://www.labins.org/survey_data/landrecords/glo_volsearch2.cfm?mvol={n:03d}"
hdr = {"User-Agent": "aoa-research (jwrampino@uchicago.edu)"}

# load checkpoint
done = set()
if logf.exists():
    with open(logf) as fh:
        done = {int(r["vol"]) for r in csv.DictReader(fh) if r["pdf_status"] == "200"}
else:
    logf.write_text("vol,page_status,pdf_url,pdf_status,bytes\n")
print(len(done), "volumes already done")

# get with retry on 5xx
def get(url):
    for i in range(3):
        r = requests.get(url, headers=hdr, timeout=60)
        if r.status_code < 500:
            return r
        time.sleep(2 * (i + 1))
    return r

# fetch pages then pdfs
for n in range(1, 401):
    if n in done:
        continue
    p = get(page.format(n=n))
    print(n, "page", p.status_code)
    url, pst, size = "", "", 0
    if p.status_code == 200:
        # find link by anchor text
        m = re.search(r'<a[^>]*href=["\']([^"\']+)["\'][^>]*>\s*All Index and Diagram Pages', p.text, re.I | re.S)
        if m:
            url = quote(urljoin(p.url, html.unescape(m.group(1))), safe=":/&%?=")   # encode spaces
            r = get(url)
            pst = str(r.status_code)
            print(n, "pdf", r.status_code, url)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                (out / f"vol{n:03d}.pdf").write_bytes(r.content)
                size = len(r.content)
            elif r.status_code == 200:
                pst = "not pdf"
        else:
            pst = "no link"
            print(n, "no pdf link")
    with open(logf, "a", newline="") as fh:   # checkpoint per volume
        csv.writer(fh).writerow([n, p.status_code, url, pst, size])
    if p.status_code >= 400 or (pst.isdigit() and int(pst) >= 400):   # stop on any 4xx
        print("stopping at", n)
        break
    time.sleep(2)   # rate limit

# summary
print(len(list(out.glob("vol*.pdf"))), "files saved")

0 volumes already done
1 page 200
1 pdf 200 https://ftp.labins.org/glo_all/Volume1_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
2 page 200
2 pdf 200 https://ftp.labins.org/glo_all/Volume2_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
3 page 200
3 pdf 200 https://ftp.labins.org/glo_all/Volume3_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
4 page 200
4 pdf 200 https://ftp.labins.org/glo_all/Volume4_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
5 page 200
5 pdf 200 https://ftp.labins.org/glo_all/Volume5_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
6 page 200
6 pdf 200 https://ftp.labins.org/glo_all/Volume6_pdf/Folder%201%20Index%20&%20Diagram_pdf.pdf
7 page 200
7 pdf 200 https://ftp.labins.org/glo_all/Volume7_pdf/Folder%20%201%20Index%20&%20Digram_pdf.pdf
8 page 200
8 pdf 200 https://ftp.labins.org/glo_all/Volume8_pdf/Folder%201%20Index%20&%20Digram_pdf.pdf
9 page 200
9 pdf 200 https://ftp.labins.org/glo_all/Volume9_pdf/Folder%20%201%20Index%20&%20Digram_pdf.pdf
10 page 200
10 pdf 200 https: